# IE212 – Lab 3: Bài tập về nhà

## Bài tập về nhà: Fecom

#### MSSV: 23521720
#### Họ tên: Nguyễn Minh Tuấn


## A. Chuẩn bị dữ liệu




### A.1. Cài PySpark 3.5.1 và tạo SparkSession

In [1]:
%pip install -q pyspark==3.5.1


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [2]:
from google.colab import files
import zipfile
import io
import os

# Upload data.zip neu chua co du lieu
if not os.path.exists("/content/data/movielens-mini/movies.txt") or \
   not os.path.exists("/content/data/fecom/Orders.csv"):

    uploaded = files.upload()

    with zipfile.ZipFile(io.BytesIO(uploaded["data.zip"])) as z:
        z.extractall("/content/")

# Kiem tra du lieu
print("MovieLens:", os.listdir("/content/data/movielens-mini"))
print("Fecom:", os.listdir("/content/data/fecom"))

ML = "/content/data/movielens-mini"
FE = "/content/data/fecom"

print("Duong dan MovieLens:", ML)
print("Duong dan Fecom:", FE)

Saving data.zip to data.zip
MovieLens: ['ratings_2.txt', 'ratings_1.txt', 'occupation.txt', 'movies.txt', 'users.txt']
Fecom: ['Customer_List.csv', 'Order_Items.csv', 'Order_Reviews.csv', 'Products.csv', 'Orders.csv']
Duong dan MovieLens: /content/data/movielens-mini
Duong dan Fecom: /content/data/fecom


In [3]:
import os
import subprocess
from pathlib import Path

# Spark 3.5.1 dùng Java 17 trên Colab.
if os.name == "posix":
    java17 = sorted(Path("/usr/lib/jvm").glob("java-17-openjdk-*"))
    if not java17:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jre-headless"], check=True)
        java17 = sorted(Path("/usr/lib/jvm").glob("java-17-openjdk-*"))
    if not java17:
        raise RuntimeError("Không tìm thấy Java 17 sau khi cài đặt.")
    os.environ["JAVA_HOME"] = str(java17[0])

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("IE212-Lab3-BTVN")
         .master("local[*]")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("PySpark:", spark.version)
assert spark.version == "3.5.1", "Hãy dùng runtime mới rồi chạy lại từ cell cài đặt."

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StringType


PySpark: 3.5.1


### A.2. Upload và kiểm tra file

In [4]:
import os
from pathlib import Path, PurePosixPath
from io import BytesIO
from zipfile import ZipFile, BadZipFile


def prepare_data(directory, required):
    target = Path(directory)
    target.mkdir(parents=True, exist_ok=True)
    missing = [name for name in required
               if not (target / name).is_file() or (target / name).stat().st_size == 0]
    if missing:
        print("Còn thiếu:", ", ".join(missing))
        print("Chọn các file dữ liệu hoặc data.zip trong cửa sổ upload.")
        try:
            from google.colab import files
        except ImportError as exc:
            raise RuntimeError("Hãy chạy notebook trên Google Colab để upload dữ liệu.") from exc
        uploaded = files.upload()
        expected = {name.casefold(): name for name in required}
        found = {}

        def remember(name, payload):
            filename = PurePosixPath(name.replace("\\", "/")).name
            canonical = expected.get(filename.casefold())
            if canonical:
                if canonical in found and found[canonical] != payload:
                    raise ValueError(f"Có nhiều bản khác nội dung của {canonical} trong lần upload.")
                found[canonical] = payload

        for name, payload in uploaded.items():
            if name.lower().endswith(".zip"):
                try:
                    with ZipFile(BytesIO(payload)) as archive:
                        for member in archive.infolist():
                            parts = PurePosixPath(member.filename.replace("\\", "/")).parts
                            if member.is_dir() or "__MACOSX" in parts:
                                continue
                            if parts and parts[-1].casefold() in expected:
                                remember(member.filename, archive.read(member))
                except BadZipFile as exc:
                    raise ValueError(f"ZIP không hợp lệ: {name}") from exc
            else:
                remember(name, payload)
        # Chỉ ghi đúng tên file cần dùng; không giữ cấu trúc lồng trong ZIP.
        for name, payload in found.items():
            (target / name).write_bytes(payload)
    present = [name for name in required
               if (target / name).is_file() and (target / name).stat().st_size > 0]
    print("Tìm thấy:", ", ".join(present))
    missing = sorted(set(required) - set(present))
    if missing:
        raise FileNotFoundError("Còn thiếu: " + ", ".join(missing)
                                + ". Chạy lại cell và chọn bổ sung các file này.")
    print("Thư mục dữ liệu:", target)
    return target

FE = "/content/data/fecom"
FE_FILES = ("Orders.csv", "Order_Items.csv", "Products.csv", "Customer_List.csv", "Order_Reviews.csv")
prepare_data(FE, FE_FILES)


Tìm thấy: Orders.csv, Order_Items.csv, Products.csv, Customer_List.csv, Order_Reviews.csv
Thư mục dữ liệu: /content/data/fecom


PosixPath('/content/data/fecom')

## B. Bài tập bắt buộc



### Bài 1. Đọc dữ liệu và kiểm tra lược đồ

In [5]:
EXPECTED_ROWS = {
    "Orders": 99441, "Order_Items": 112650, "Products": 32951,
    "Customer_List": 102727, "Order_Reviews": 99223,
}
REQUIRED_COLUMNS = {
    "Orders": ("Order_ID", "Customer_Trx_ID", "Order_Status", "Order_Purchase_Timestamp",
               "Order_Approved_At", "Order_Delivered_Carrier_Date",
               "Order_Delivered_Customer_Date", "Order_Estimated_Delivery_Date"),
    "Order_Items": ("Order_ID", "Order_Item_ID", "Product_ID", "Seller_ID", "Shipping_Limit_Date",
                    "Price", "Freight_Value"),
    "Products": ("Product_ID", "Product_Category_Name"),
    "Customer_List": ("Customer_Trx_ID", "Subscriber_ID", "Customer_Country",
                      "Subscribe_Date", "First_Order_Date"),
    "Order_Reviews": ("Review_ID", "Order_ID", "Review_Score", "Review_Creation_Date",
                      "Review_Answer_Timestamp"),
}
tables = {}
row_counts = {}
count_rows = []
for name, expected in EXPECTED_ROWS.items():
    options = dict(sep=";", header=True, inferSchema=True, encoding="UTF-8", mode="FAILFAST")
    if name == "Order_Reviews":
        options.update(multiLine=True, escape='"')
    df = spark.read.options(**options).csv(str(Path(FE) / f"{name}.csv"))
    df = df.toDF(*[column.lstrip("\ufeff").strip() for column in df.columns])
    missing = sorted(set(REQUIRED_COLUMNS[name]) - set(df.columns))
    assert not missing, f"{name}: thiếu cột {missing}; kiểm tra dấu phân tách và BOM."
    print(f"\n{name}")
    df.printSchema()
    tables[name] = df.cache()
    row_counts[name] = df.count()
    count_rows.append((name, row_counts[name], expected))

spark.createDataFrame(count_rows, "Bang STRING, So_dong LONG, Moc_doi_chieu LONG").show(truncate=False)
assert all(row_counts[name] == expected for name, expected in EXPECTED_ROWS.items()), (
    "Số dòng không khớp. Kiểm tra file, sep=';', multiLine=True và escape cho Order_Reviews."
)

# Kiểu dữ liệu thực tế do Spark suy ra; chỉ tạo bảng từ metadata lược đồ.
schema_rows = [(name, field.name, field.dataType.simpleString())
               for name, df in tables.items() for field in df.schema.fields]
spark.createDataFrame(schema_rows, "Bang STRING, Cot STRING, Kieu_suy_ra STRING") \
    .show(len(schema_rows), truncate=False)



Orders
root
 |-- Order_ID: string (nullable = true)
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Purchase_Timestamp: timestamp (nullable = true)
 |-- Order_Approved_At: timestamp (nullable = true)
 |-- Order_Delivered_Carrier_Date: timestamp (nullable = true)
 |-- Order_Delivered_Customer_Date: timestamp (nullable = true)
 |-- Order_Estimated_Delivery_Date: timestamp (nullable = true)


Order_Items
root
 |-- Order_ID: string (nullable = true)
 |-- Order_Item_ID: integer (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Seller_ID: string (nullable = true)
 |-- Shipping_Limit_Date: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Freight_Value: double (nullable = true)


Products
root
 |-- Product_ID: string (nullable = true)
 |-- Product_Category_Name: string (nullable = true)
 |-- Product_Weight_Gr: integer (nullable = true)
 |-- Product_Length_Cm: integer (nullable = true)
 |-- Product_Height

In [6]:
# Chuẩn hóa thời gian và kiểm tra khóa
TIME_COLUMNS = {
    "Orders": ("Order_Purchase_Timestamp", "Order_Approved_At", "Order_Delivered_Carrier_Date",
               "Order_Delivered_Customer_Date", "Order_Estimated_Delivery_Date"),
    "Order_Items": ("Shipping_Limit_Date",),
    "Customer_List": ("Subscribe_Date", "First_Order_Date"),
    "Order_Reviews": ("Review_Creation_Date", "Review_Answer_Timestamp"),
}
string_time_rows = []
parse_rows = []
for name, columns in TIME_COLUMNS.items():
    df = tables[name]
    pattern = "yyyy-MM-dd" if name == "Customer_List" else "yyyy-MM-dd HH:mm"
    checks = []
    for column in columns:
        if isinstance(df.schema[column].dataType, StringType):
            string_time_rows.append((name, column, pattern))
            checks.append(F.count(F.when(
                F.col(column).isNotNull() & F.to_timestamp(column, pattern).isNull(), 1
            )).alias(column))
    if checks:
        failures = df.agg(*checks).first().asDict()  # Một dòng thống kê, không lấy dữ liệu gốc.
        parse_rows.extend((name, column, int(n)) for column, n in failures.items())
        assert all(n == 0 for n in failures.values()), f"{name}: lỗi định dạng thời gian {failures}"
    for column in columns:
        if isinstance(df.schema[column].dataType, StringType):
            df = df.withColumn(column, F.to_timestamp(column, pattern))
        else:
            df = df.withColumn(column, F.col(column).cast("timestamp"))
    tables[name] = df

spark.createDataFrame(string_time_rows, "Bang STRING, Cot_string STRING, Dinh_dang STRING") \
    .show(len(string_time_rows) or 1, truncate=False)
spark.createDataFrame(parse_rows, "Bang STRING, Cot STRING, So_loi_chuyen LONG") \
    .show(len(parse_rows) or 1, truncate=False)

orders = tables["Orders"].cache()
items = tables["Order_Items"]
products = tables["Products"]
customers = tables["Customer_List"]
reviews = tables["Order_Reviews"]

# Những bảng nối làm chiều tra cứu phải duy nhất theo khóa.
for label, df, key in [("Orders", orders, "Order_ID"), ("Products", products, "Product_ID")]:
    stats = df.agg(F.count("*").alias("rows"), F.countDistinct(key).alias("keys")).first()
    assert stats["rows"] == stats["keys"], f"{label}: khóa trùng hoặc NULL."
customer_lookup = customers.where(F.col("Customer_Trx_ID").isNotNull()) \
    .select("Customer_Trx_ID", "Subscriber_ID", "Customer_Country")
assert customer_lookup.count() == customer_lookup.select("Customer_Trx_ID").distinct().count()
assert customer_lookup.where(F.col("Subscriber_ID").isNull()).count() == 0


+----+----------+---------+
|Bang|Cot_string|Dinh_dang|
+----+----------+---------+
+----+----------+---------+

+----+---+-------------+
|Bang|Cot|So_loi_chuyen|
+----+---+-------------+
+----+---+-------------+



### Bài 2. Tổng số đơn hàng, khách hàng và người bán


In [7]:
purchasing_customers = customer_lookup.join(
    orders.select("Customer_Trx_ID").distinct(), "Customer_Trx_ID", "left_semi"
)
summary = (orders.agg(F.count("*").alias("so_don_hang"))
           .crossJoin(purchasing_customers.agg(F.countDistinct("Subscriber_ID").alias("so_khach_hang")))
           .crossJoin(items.agg(F.countDistinct("Seller_ID").alias("so_nguoi_ban"))))
summary.show()
customers.agg(F.count("*").alias("so_dong_dang_ky"),
              F.count(F.when(F.col("Customer_Trx_ID").isNull(), 1)).alias("chua_co_giao_dich")).show()


+-----------+-------------+------------+
|so_don_hang|so_khach_hang|so_nguoi_ban|
+-----------+-------------+------------+
|      99441|        96096|        3095|
+-----------+-------------+------------+

+---------------+-----------------+
|so_dong_dang_ky|chua_co_giao_dich|
+---------------+-----------------+
|         102727|             3286|
+---------------+-----------------+



### Bài 3. Số đơn hàng theo quốc gia



In [8]:
orders_customers = orders.join(customer_lookup, "Customer_Trx_ID", "left").cache()
join_check = orders_customers.agg(
    F.count("*").alias("so_dong_sau_join"), F.countDistinct("Order_ID").alias("so_don_phan_biet"),
    F.count(F.when(F.col("Subscriber_ID").isNull(), 1)).alias("khong_khop_khach")
)
join_check.show()
check = join_check.first()
assert check["so_dong_sau_join"] == row_counts["Orders"] == check["so_don_phan_biet"]
assert check["khong_khop_khach"] == 0
by_country = orders_customers.groupBy("Customer_Country").agg(F.count("*").alias("so_don_hang"))
by_country.orderBy(F.desc("so_don_hang"), "Customer_Country").show(10, truncate=False)
by_country.agg(F.count("*").alias("so_quoc_gia"), F.sum("so_don_hang").alias("tong_so_don")).show()
assert by_country.agg(F.sum("so_don_hang")).first()[0] == row_counts["Orders"]


+----------------+----------------+----------------+
|so_dong_sau_join|so_don_phan_biet|khong_khop_khach|
+----------------+----------------+----------------+
|           99441|           99441|               0|
+----------------+----------------+----------------+

+----------------+-----------+
|Customer_Country|so_don_hang|
+----------------+-----------+
|Germany         |41754      |
|France          |12848      |
|Netherlands     |11629      |
|Belgium         |5464       |
|Austria         |5043       |
|Switzerland     |3640       |
|United Kingdom  |3382       |
|Poland          |2139       |
|Czechia         |2034       |
|Italy           |2025       |
+----------------+-----------+
only showing top 10 rows

+-----------+-----------+
|so_quoc_gia|tong_so_don|
+-----------+-----------+
|         27|      99441|
+-----------+-----------+



### Bài 4. Số đơn hàng theo năm và tháng



In [9]:
assert orders.where(F.col("Order_Purchase_Timestamp").isNull()).count() == 0
by_month = (orders.withColumn("nam", F.year("Order_Purchase_Timestamp"))
            .withColumn("thang", F.month("Order_Purchase_Timestamp"))
            .groupBy("nam", "thang").agg(F.count("*").alias("so_don_hang")))
by_month.orderBy(F.asc("nam"), F.desc("thang")).show(by_month.count(), truncate=False)
orders.agg(F.min("Order_Purchase_Timestamp").alias("bat_dau"),
           F.max("Order_Purchase_Timestamp").alias("ket_thuc")).show(truncate=False)
assert by_month.agg(F.sum("so_don_hang")).first()[0] == row_counts["Orders"]


+----+-----+-----------+
|nam |thang|so_don_hang|
+----+-----+-----------+
|2022|12   |1          |
|2022|10   |324        |
|2022|9    |4          |
|2023|12   |5673       |
|2023|11   |7544       |
|2023|10   |4631       |
|2023|9    |4285       |
|2023|8    |4331       |
|2023|7    |4026       |
|2023|6    |3245       |
|2023|5    |3700       |
|2023|4    |2404       |
|2023|3    |2682       |
|2023|2    |1780       |
|2023|1    |800        |
|2024|10   |4          |
|2024|9    |16         |
|2024|8    |6512       |
|2024|7    |6292       |
|2024|6    |6167       |
|2024|5    |6873       |
|2024|4    |6939       |
|2024|3    |7211       |
|2024|2    |6728       |
|2024|1    |7269       |
+----+-----+-----------+

+-------------------+-------------------+
|bat_dau            |ket_thuc           |
+-------------------+-------------------+
|2022-09-04 21:15:00|2024-10-17 17:30:00|
+-------------------+-------------------+



### Bài 5. Điểm đánh giá trung bình và phân bố theo mức


In [10]:
bad_score = (F.col("Review_Score").isNull() | F.isnan(F.col("Review_Score").cast("double"))
             | ~F.col("Review_Score").isin(1, 2, 3, 4, 5))
score_audit = reviews.agg(
    F.count("*").alias("so_danh_gia"),
    F.count(F.when(F.col("Review_Score").isNull(), 1)).alias("so_NULL"),
    F.count(F.when(bad_score, 1)).alias("so_khong_hop_le")
)
score_audit.show()
assert score_audit.first()["so_khong_hop_le"] == 0, (
    "Review_Score có lỗi: kiểm tra multiLine=True, escape và file nguồn. Chưa loại dòng nào."
)
reviews.groupBy("Review_Score").agg(F.count("*").alias("so_luong")) \
    .orderBy("Review_Score").show()
reviews.agg(F.round(F.avg("Review_Score"), 2).alias("diem_trung_binh"),
            F.count("*").alias("so_danh_gia")).show()


+-----------+-------+---------------+
|so_danh_gia|so_NULL|so_khong_hop_le|
+-----------+-------+---------------+
|      99223|      0|              0|
+-----------+-------+---------------+

+------------+--------+
|Review_Score|so_luong|
+------------+--------+
|           1|   11424|
|           2|    3151|
|           3|    8179|
|           4|   19141|
|           5|   57328|
+------------+--------+

+---------------+-----------+
|diem_trung_binh|so_danh_gia|
+---------------+-----------+
|           4.09|      99223|
+---------------+-----------+



### Bài 6. Xếp hạng người bán



In [11]:
items_money = (items.withColumn("Price", F.col("Price").cast("decimal(20,2)"))
               .withColumn("Freight_Value", F.col("Freight_Value").cast("decimal(20,2)"))
               .withColumn("gia_tri_dong", F.col("Price") + F.col("Freight_Value")))
assert items_money.where(F.col("gia_tri_dong").isNull() | F.col("Seller_ID").isNull()).count() == 0
seller_sales = items_money.groupBy("Seller_ID").agg(
    F.sum("gia_tri_dong").alias("doanh_thu"), F.countDistinct("Order_ID").alias("so_don")
)
w_seller = Window.orderBy(F.desc("doanh_thu"))
seller_ranking = seller_sales.withColumn("hang", F.rank().over(w_seller))
seller_ranking.orderBy(F.desc("doanh_thu"), "Seller_ID") \
    .select("hang", "Seller_ID", F.round("doanh_thu", 2).alias("doanh_thu"), "so_don").show(5)
print("Tổng số người bán trong bảng xếp hạng:", seller_ranking.count())
assert seller_ranking.count() == items.select("Seller_ID").distinct().count()


+----+--------------------+---------+------+
|hang|           Seller_ID|doanh_thu|so_don|
+----+--------------------+---------+------+
|   1|4869f7a5dfa277a7d...|249640.70|  1132|
|   2|7c67e1448b00f6e96...|239536.44|   982|
|   3|53243585a1d6dc264...|235856.68|   358|
|   4|4a3ca9315b744ce9f...|235539.96|  1806|
|   5|fa1c13f2614d7b5c4...|204084.73|   585|
+----+--------------------+---------+------+
only showing top 5 rows

Tổng số người bán trong bảng xếp hạng: 3095


## C. Bài tập tự chọn



### Bài 7. Doanh thu năm 2024 theo danh mục sản phẩm

In [12]:
orders_2024 = orders.where(F.year("Order_Purchase_Timestamp") == 2024).select("Order_ID")
items_2024 = items_money.join(orders_2024, "Order_ID", "inner").cache()
assert items_2024.join(products, "Product_ID", "left_anti").count() == 0
category_items = items_2024.join(products.select("Product_ID", "Product_Category_Name"),
                                 "Product_ID", "left")
assert category_items.count() == items_2024.count()
by_category = category_items.groupBy("Product_Category_Name").agg(
    F.sum("gia_tri_dong").alias("doanh_thu")
)
by_category.orderBy(F.desc("doanh_thu"), "Product_Category_Name") \
    .select("Product_Category_Name", F.round("doanh_thu", 2).alias("doanh_thu")) \
    .show(by_category.count(), truncate=False)
items_2024.agg(F.count("*").alias("so_dong_hang_2024"),
               F.round(F.sum("gia_tri_dong"), 2).alias("tong_doanh_thu_2024")).show()
by_category.agg(F.count("*").alias("so_danh_muc"),
                F.round(F.sum("doanh_thu"), 2).alias("tong_doanh_thu_danh_muc")).show()
assert by_category.agg(F.sum("doanh_thu")).first()[0] == items_2024.agg(F.sum("gia_tri_dong")).first()[0]


+---------------------------------------+---------+
|Product_Category_Name                  |doanh_thu|
+---------------------------------------+---------+
|Health_Beauty                          |885191.12|
|Watches_Gifts                          |771986.75|
|Bed_Bath_Table                         |650794.70|
|Sports_Leisure                         |621999.34|
|Computers_Accessories                  |594771.04|
|Housewares                             |491576.96|
|Furniture_Decor                        |476466.13|
|Auto                                   |404210.57|
|Baby                                   |299052.56|
|Cool_Stuff                             |273910.05|
|Garden_Tools                           |259068.32|
|Telephony                              |217452.13|
|Perfumery                              |204562.54|
|Toys                                   |200634.07|
|Office_Furniture                       |181745.73|
|Stationery                             |164743.85|
|Pet_Shop   

### Bài 8. Sản phẩm bán chạy nhất và điểm đánh giá trung bình

In [13]:
reviews_per_order = reviews.groupBy("Order_ID").agg(
    F.avg("Review_Score").alias("diem_don"), F.count("*").alias("so_danh_gia_don")
)
reviews_per_order.agg(
    F.count(F.when(F.col("so_danh_gia_don") > 1, 1)).alias("so_don_nhieu_danh_gia"),
    F.sum(F.col("so_danh_gia_don") - 1).alias("so_dong_tang_neu_noi_truc_tiep")
).show()
orders.join(reviews_per_order.select("Order_ID"), "Order_ID", "left_anti") \
    .agg(F.count("*").alias("so_don_khong_danh_gia")).show()

rated_items = items.join(reviews_per_order, "Order_ID", "left")
assert rated_items.count() == row_counts["Order_Items"], "Join đã làm đổi số dòng hàng."
product_sales = rated_items.groupBy("Product_ID").agg(
    F.count("*").alias("so_luong"), F.avg("diem_don").alias("diem_tb"),
    F.count("diem_don").alias("so_luong_co_danh_gia")
)
best_products = product_sales.join(products.select("Product_ID", "Product_Category_Name"),
                                    "Product_ID", "left")
assert best_products.count() == product_sales.count()
best_products.orderBy(F.desc("so_luong"), "Product_ID").select(
    "Product_ID", F.col("Product_Category_Name").alias("danh_muc"), "so_luong",
    F.round("diem_tb", 2).alias("diem_tb"), "so_luong_co_danh_gia"
).show(5)
assert product_sales.agg(F.sum("so_luong")).first()[0] == row_counts["Order_Items"]


+---------------------+------------------------------+
|so_don_nhieu_danh_gia|so_dong_tang_neu_noi_truc_tiep|
+---------------------+------------------------------+
|                  547|                           551|
+---------------------+------------------------------+

+---------------------+
|so_don_khong_danh_gia|
+---------------------+
|                  769|
+---------------------+

+--------------------+---------------+--------+-------+--------------------+
|          Product_ID|       danh_muc|so_luong|diem_tb|so_luong_co_danh_gia|
+--------------------+---------------+--------+-------+--------------------+
|aca2eb7d00ea1a7b8...|Furniture_Decor|     527|   4.02|                 524|
|99a4788cb24856965...| Bed_Bath_Table|     488|    3.9|                 479|
|422879e10f4668299...|   Garden_Tools|     484|   3.96|                 483|
|389d119b48cf3043d...|   Garden_Tools|     392|   4.12|                 391|
|368c6c730842d7801...|   Garden_Tools|     388|   3.92|         